# Assemble per-run summary and ph csv files

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
path_main = Path("../simulations")

path_data = path_main / "20251119"

path_save = path_main / "20251119_summary"
if not path_save.exists():
    path_save.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(search_type, cr, br, pm_agent, pm_truth):
    dates = ["20251119", "20251120"]
    files = []
    for d in dates:
        folder = f"{d}_{search_type}_pmaxRepeat_canvasRadius{cr}_beamRadius{br}_pmAgent{pm_agent:.0e}_pfaAgent{pm_agent:.0e}_beamRadius{br}_pmTruth{pm_truth:.0e}_pfaTruth{pm_truth:.0e}"
        files += sorted(list((path_data / folder).glob("*_summary.csv")))

    data_list = []
    for f in files:
        data_list.append(pd.read_csv(f, index_col=0))
    df = pd.concat(data_list).reset_index(drop=True)
    return df

In [4]:
def load_ph(search_type, cr, br, pm_agent, pm_truth):
    dates = ["20251119", "20251120"]
    files_ph = []
    files_dk_max_cube = []
    for d in dates:
        folder = f"{d}_{search_type}_pmaxRepeat_canvasRadius{cr}_beamRadius{br}_pmAgent{pm_agent:.0e}_pfaAgent{pm_agent:.0e}_beamRadius{br}_pmTruth{pm_truth:.0e}_pfaTruth{pm_truth:.0e}"
        files_ph += sorted(list((path_data / folder).glob("*_ph.csv")))
        files_dk_max_cube += sorted(list((path_data / folder).glob("*_dk_max_cube.nc")))

    # Sanity check the file number matches
    assert len(files_ph) == len(files_dk_max_cube), (
        f"Number of _ph files ({len(files_ph)}) does not match number of "
        f"_dk_max_cube files ({len(files_dk_max_cube)}) for "
        f"cr={cr}, br={br}, pm_agent={pm_agent:.0e}, pm_truth={pm_truth:.0e}"
    )

    # Assemble data into lists
    p_list = []
    h_list = []
    target_list = []
    dk_max_cube_list = []
    for f_ph, f_dk_max_cube in zip(files_ph, files_dk_max_cube):
        df = pd.read_csv(f_ph, index_col=0)
        p_list.append(df["p_max_all"])
        h_list.append(df["h_actual_all"])

        ds = xr.open_dataset(f_dk_max_cube)
        target_list.append(ds["target_cube"].values)
        dk_max_cube_list.append(ds["max_dk_cube"].values)

    # If less than 1000 runs, print out missing run
    if len(h_list) < 1000:
        run_nums = []
        for f in files_ph:
            a = re.match(r"infotaxis_(\d{4})_ph.csv", f.name)
            run_nums.append(int(a.groups()[0]))
        missing_runs = set(range(1, 1001)) - set(run_nums)
        print(f"Missing runs for cr={cr}, br={br}, pm_agent={pm_agent:.0e}, pm_truth={pm_truth:.0e}: {missing_runs}")

    # Pad ragged list to array
    len_max = max(len(h) for h in h_list)
    h_actual_pad = np.array([h.tolist() + [np.nan] * (len_max - len(h)) for h in h_list])
    p_max_pad = np.array([p.tolist() + [np.nan] * (len_max - len(p)) for p in p_list])
    target_cube_pad = np.array(target_list)
    dk_max_cube_pad = np.array([np.vstack((d, np.nan * np.ones((len_max - len(d), 3)))) for d in dk_max_cube_list])

    ds = xr.Dataset(
        data_vars=dict(
            h_actual=(["run", "ping"], h_actual_pad),
            p_max=(["run", "ping"], p_max_pad),
            target_cube=(["run", "cube_dim"], target_cube_pad),
            dk_max_cube=(["run", "ping", "cube_dim"], dk_max_cube_pad),
        ),
        coords=dict(
            run=("run", range(1, len(h_list) + 1)),
            ping=("ping", range(len_max)),
            cube_dim=("cube_dim", ["1", "2", "3"]),
        )
    )
    return ds 

## PM=PFA sweep

In [5]:
cr_all = [5, 10]
br_all = [1, 2]
pm_agent_all = [0.001]
pm_truth_all = [0.005, 0.01, 0.02, 0.03, 0.04, 0.05]

In [6]:
# infotaxis ph data
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for pm_agent in pm_agent_all:
            for pm_truth in pm_truth_all:
                print(f"cr={cr}, br={br}, pm_agent={pm_agent:.0e}, pm_truth={pm_truth:.0e}")
                ds = load_ph(search_type, cr=cr, br=br, pm_agent=pm_agent, pm_truth=pm_truth)
                ds.to_netcdf(path_save / f"{search_type}_cr{cr}_br{br}_pmAgent{pm_agent:.0e}_pmTruth{pm_truth:.0e}_ph.nc")

cr=5, br=1, pm_agent=1e-03, pm_truth=5e-03
cr=5, br=1, pm_agent=1e-03, pm_truth=1e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=2e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=3e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=4e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=5e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=5e-03
cr=5, br=2, pm_agent=1e-03, pm_truth=1e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=2e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=3e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=4e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=5e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=5e-03
cr=10, br=1, pm_agent=1e-03, pm_truth=1e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=2e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=3e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=4e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=5e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=5e-03
cr=10, br=2, pm_agent=1e-03, pm_truth=1e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=2e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=3e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=4e-02


In [7]:
# infotaxis summary
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for pm_agent in pm_agent_all:
            for pm_truth in pm_truth_all:
                print(f"cr={cr}, br={br}, pm_agent={pm_agent:.0e}, pm_truth={pm_truth:.0e}")
                df = load_summary(search_type, cr=cr, br=br, pm_agent=pm_agent, pm_truth=pm_truth)
                df.to_csv(path_save / f"{search_type}_cr{cr}_br{br}_pmAgent{pm_agent:.0e}_pmTruth{pm_truth:.0e}_summary.csv")

cr=5, br=1, pm_agent=1e-03, pm_truth=5e-03
cr=5, br=1, pm_agent=1e-03, pm_truth=1e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=2e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=3e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=4e-02
cr=5, br=1, pm_agent=1e-03, pm_truth=5e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=5e-03
cr=5, br=2, pm_agent=1e-03, pm_truth=1e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=2e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=3e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=4e-02
cr=5, br=2, pm_agent=1e-03, pm_truth=5e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=5e-03
cr=10, br=1, pm_agent=1e-03, pm_truth=1e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=2e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=3e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=4e-02
cr=10, br=1, pm_agent=1e-03, pm_truth=5e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=5e-03
cr=10, br=2, pm_agent=1e-03, pm_truth=1e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=2e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=3e-02
cr=10, br=2, pm_agent=1e-03, pm_truth=4e-02
